# Continuous Glucose Monitoring (CGM) Report

Ambulatory Glucose Profile for the **launched** patient, sourced from the JupyterHealth
Exchange. This notebook is rendered by Voilà as the provider-facing app: the EHR launch
supplies the patient, the cells below compute and draw the report. Code input is hidden
in the rendered view (`strip_sources`), so only the report shows.

In [ ]:
# ============================================================================
# SCAFFOLD — launch context → JHE token → MRN → this patient's glucose frame.
# Usually untouched. Edit the ANALYTICS cells below, not this one.
# ============================================================================
import pandas as pd
from jupyterhealth_client import Code
from provider_app import launch_context, jhe_auth, patient_resolver

pd.options.mode.chained_assignment = None

ctx = launch_context.current()                       # SMART launch (Medplum) → patient MRN
client = jhe_auth.client_for_launch(ctx)             # JHE client (token minted from the SMART id_token exchange)

# Resolve the launched patient in JHE and pull glucose. If the JHE account isn't
# authorized to see this patient (JupyterHealth RBAC), patient_resolver raises
# PatientNotInJHE — we catch it (and any other load error) so the dashboard shows a
# clean message and reveals NO patient info, instead of a traceback. None on success.
access_note = None
patient_id = None
df = pd.DataFrame()
try:
    patient_id = patient_resolver.resolve_patient(ctx.patient_mrn, client)
    df = client.list_observations_df(patient_id=patient_id, code=Code.BLOOD_GLUCOSE, limit=10_000)
except patient_resolver.PatientNotInJHE:
    access_note = (
        "You don't have access to this patient's data in JupyterHealth. They may not be "
        "enrolled, or your JupyterHealth account isn't authorized to view them — access "
        "is governed by JupyterHealth, separately from your EHR permissions."
    )
except Exception as _e:
    access_note = f"Couldn't load JupyterHealth data — {type(_e).__name__}: {_e}"

# Tidy CGM frame, normalized to mg/dL (empty when there's no data / no access). JHE
# datasets label the unit variously ("mg/dL", "MGDL"); convert mmol/L (×18.0182).
if df.empty:
    cgm = df
else:
    cgm = df.loc[:, ["blood_glucose_value", "effective_time_frame_date_time_local"]].copy()
    _unit = df["blood_glucose_unit"].astype(str).str.upper().str.replace(r"[^A-Z]", "", regex=True)
    cgm.loc[_unit.values == "MMOLL", "blood_glucose_value"] *= 18.0182
    cgm["effective_time_frame_date_time_local"] = pd.to_datetime(
        cgm["effective_time_frame_date_time_local"]
    )
    cgm = cgm.sort_values("effective_time_frame_date_time_local")

In [ ]:
# ======== PATIENT HEADER ========
import requests as _rq
from IPython.display import Markdown, display


def md(text):
    display(Markdown(text))


try:
    me = client.get_user()
    practitioner = f"{me['firstName']} {me['lastName']} ({me['email']})"
except Exception:
    practitioner = "(unknown)"

if access_note:
    # JHE has NOT authorized this provider for this patient. Reveal nothing
    # patient-identifying on the JupyterHealth surface — only the access notice.
    md("# CGM Report")
    md(f"Signed in as **{practitioner}**")
    md(f"> 🔒 **{access_note}**")
else:
    # Authorized: demographics come from the EHR's FHIR Patient (system of record for
    # who the patient is). ctx carries the EHR token + FHIR base.
    try:
        _p = _rq.get(
            f"{ctx.fhir_base.rstrip('/')}/Patient/{ctx.fhir_patient_id}",
            headers={"Authorization": f"Bearer {ctx.access_token}"},
            timeout=15,
        ).json()
        _nm = (_p.get("name") or [{}])[0]
        _name = (" ".join(_nm.get("given", [])) + " " + _nm.get("family", "")).strip() or "(unnamed)"
        _dob = _p.get("birthDate", "—")
        _sex = (_p.get("gender") or "—").title()
        _contact = next((t.get("value") for t in _p.get("telecom", []) if t.get("value")), "")
    except Exception:
        _name, _dob, _sex, _contact = "(demographics unavailable)", "—", "—", ""

    md(f"## {_name}")
    _line = f"**DOB** {_dob}  ·  **Sex** {_sex}  ·  **MRN** `{ctx.patient_mrn}`  ·  **JHE patient** {patient_id}"
    if _contact:
        _line += f"  ·  {_contact}"
    md(_line)
    md(f"Reviewing practitioner: **{practitioner}**")
    if cgm.empty:
        md("> **No glucose data for this patient.** They appear enrolled, but no blood-glucose observations were found.")
    else:
        span = (
            f"{cgm.effective_time_frame_date_time_local.min():%Y-%m-%d} → "
            f"{cgm.effective_time_frame_date_time_local.max():%Y-%m-%d}"
        )
        md(f"{len(cgm):,} glucose readings · {span}")

In [ ]:
# ======== GLYCEMIC METRICS (GMI, CV, Time-in-Range, GRI) ========
if not cgm.empty:
    g = cgm.blood_glucose_value
    total = len(g)
    mean_glucose = g.mean()
    cv = g.std() / mean_glucose * 100
    gmi = 3.31 + 0.02392 * mean_glucose

    vlow = (g < 54).mean() * 100
    low = ((g >= 54) & (g < 70)).mean() * 100
    target = ((g >= 70) & (g <= 180)).mean() * 100
    high = ((g > 180) & (g <= 250)).mean() * 100
    vhigh = (g > 250).mean() * 100

    # Glycemia Risk Index (0 best → 100 worst), capped at 100.
    gri = min(3.0 * (vlow + 0.8 * low) + 1.6 * (vhigh + 0.5 * high), 100)

    md(
        "| Metric | Value |\n|---|---|\n"
        f"| Mean glucose | {mean_glucose:.0f} mg/dL |\n"
        f"| Glucose Management Indicator (GMI) | {gmi:.1f}% |\n"
        f"| Coefficient of variation (CV) | {cv:.0f}% |\n"
        f"| Time in range (70–180) | {target:.0f}% |\n"
        f"| Glycemia Risk Index (GRI) | {gri:.0f} |"
    )

In [ ]:
# ======== AMBULATORY GLUCOSE PROFILE (AGP) ========
# All readings collapsed onto a single 24-hour day, shown as percentile bands.
if not cgm.empty:
    import matplotlib.pyplot as plt

    t = cgm.effective_time_frame_date_time_local
    cgm["minute_of_day"] = t.dt.hour * 60 + t.dt.minute
    cgm["time_bin"] = (cgm.minute_of_day // 5) * 5

    agp = (
        cgm.groupby("time_bin").blood_glucose_value
        .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        .unstack()
    )
    agp.columns = ["p5", "p25", "p50", "p75", "p95"]
    agp = agp.reset_index()
    agp["hour"] = agp.time_bin / 60

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axhspan(70, 180, color="#CDECCD", alpha=0.5, label="Target 70–180")
    ax.fill_between(agp.hour, agp.p5, agp.p95, color="#D8E3E7", alpha=0.6, label="5–95%")
    ax.fill_between(agp.hour, agp.p25, agp.p75, color="#B0C4DE", alpha=0.7, label="25–75%")
    ax.plot(agp.hour, agp.p50, color="green", lw=2, label="Median")
    ax.set_title("Ambulatory Glucose Profile (AGP)")
    ax.set_xlabel("Time of day")
    ax.set_ylabel("Glucose (mg/dL)")
    ax.set_xticks([0, 6, 12, 18, 24])
    ax.set_xticklabels(["12 AM", "6 AM", "12 PM", "6 PM", "12 AM"])
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 300)
    ax.set_yticks([54, 70, 180, 250, 300])
    ax.legend(loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# ======== TIME IN RANGE (stacked bar) ========
if not cgm.empty:
    bands = {
        "Very Low (<54)": (vlow, "#a00000"),
        "Low (54–69)": (low, "#f44336"),
        "Target (70–180)": (target, "#9ccc9c"),
        "High (181–250)": (high, "#fdbe85"),
        "Very High (>250)": (vhigh, "#fd8d3c"),
    }
    fig, ax = plt.subplots(figsize=(3, 6))
    bottom = 0
    for label, (pct, color) in bands.items():
        ax.bar(0, pct, bottom=bottom, color=color, width=0.6, edgecolor="white")
        if pct > 3:
            ax.text(0.45, bottom + pct / 2, f"{pct:.0f}%  {label}", va="center", fontsize=9)
        bottom += pct
    ax.set_ylim(0, 100)
    ax.set_xlim(-0.4, 2.2)
    ax.set_xticks([])
    ax.set_ylabel("% of readings")
    ax.set_title("Time in Range")
    plt.show()